In [8]:
import pandas as pd
from pathlib import Path

# 1. Define the repo root and path to your millage table
ROOT = Path().resolve().parents[1]  
MILLAGE_FP = ROOT / 'data' / 'Township_Millage_Table.csv'
OUTPUT_DIR = ROOT / 'outputs' / 'plots'

In [9]:
# 2 Calculate Average Percent Change by Property Class
df = pd.read_csv(MILLAGE_FP)

collapsed = (
    df.groupby(["TaxYear", "PropertyClass"], as_index=False)
      .agg(FireRate=("FireRate", "mean"))
      .sort_values(["PropertyClass", "TaxYear"])
)

# Year-over-year % change by PropertyClass
collapsed["FireRate_pct_change"] = (
    collapsed.groupby("PropertyClass")["FireRate"].pct_change() * 100
)

# Average % change by PropertyClass
summary = (
    collapsed.dropna(subset=["FireRate_pct_change"])
             .groupby("PropertyClass", as_index=False)
             .agg(avg_fire_rate_pct_change=("FireRate_pct_change", "mean"))
)

# Print results
print("Yearly FireRate % Change by PropertyClass:")
print(collapsed.to_string(index=False))
print("\nAverage FireRate % Change by PropertyClass:")
print(summary.to_string(index=False))

Yearly FireRate % Change by PropertyClass:
 TaxYear PropertyClass  FireRate  FireRate_pct_change
    2014        ComInd 10.985198                  NaN
    2015        ComInd  9.967344            -9.265686
    2016        ComInd 10.105672             1.387812
    2017        ComInd  9.783333            -3.189684
    2018        ComInd  9.708166            -0.768317
    2019        ComInd  9.817309             1.124239
    2020        ComInd  8.882938            -9.517588
    2021        ComInd  9.323242             4.956738
    2022        ComInd 11.072165            18.758743
    2023        ComInd  9.610293           -13.203127
    2024        ComInd  9.656662             0.482493
    2014        ResAgr  9.968231                  NaN
    2015        ResAgr  9.204195            -7.664710
    2016        ResAgr  9.187970            -0.176278
    2017        ResAgr  8.219631           -10.539205
    2018        ResAgr  8.223704             0.049552
    2019        ResAgr  8.107110       

In [10]:
import pandas as pd

# --- 1) Input data (as you provided) ----------------------------------------
taxable_data = {
    ("Jefferson Unincorporated", "ResAgr"): {
        2014: (407_807_070, 2_731_540),
        2015: (389_535_970, 2_893_880),
        2016: (394_781_780, 3_114_460),
        2017: (448_618_560, 3_264_290),
        2018: (457_955_400, 3_368_530),
        2019: (477_750_550, 5_763_030),
        2020: (568_049_370, 3_308_040),
        2021: (585_038_010, 3_641_900),
        2022: (576_718_700, 3_603_720),
        2023: (768_096_650, 4_056_820),
        2024: (763_057_600, 3_177_510),
    },
    ("Jefferson Unincorporated", "ComInd"): {
        2014: (18_498_090, 36_028_230),
        2015: (8_989_220, 11_740_430),
        2016: (8_814_320, 8_106_700),
        2017: (9_816_490, 9_176_010),
        2018: (15_470_630, 9_214_410),
        2019: (11_671_440, 9_214_410),
        2020: (12_940_970, 10_286_530),
        2021: (13_639_040, 10_633_240),
        2022: (13_244_310, 10_148_800),
        2023: (26_236_970, 12_954_810),
        2024: (16_847_860, 12_925_410),
    },
    ("Gahanna", "ResAgr"): {
        2014: (2_705_600, 98_970),
        2015: (2_766_850, 98_970),
        2016: (2_755_050, 98_970),
        2017: (2_730_400, 99_530),
        2018: (2_823_610, 99_530),
        2019: (3_152_540, 99_530),
        2020: (4_121_040, 89_370),
        2021: (4_182_260, 8_990),
        2022: (4_165_320, 0),
        2023: (5_379_930, 0),
        2024: (5_347_560, 0),
    },
    ("Gahanna", "ComInd"): {
        2014: (457_120, 1_650),
        2015: (460_340, 1_650),
        2016: (651_580, 1_650),
        2017: (626_240, 1_650),
        2018: (1_980_250, 1_650),
        2019: (2_238_700, 1_650),
        2020: (2_511_240, 1_890),
        2021: (3_015_130, 1_890),
        2022: (3_514_270, 603_650),
        2023: (8_013_600, 830_240),
        2024: (5_418_120, 546_950),
    },
    ("Reynoldsburg", "ResAgr"): {
        2014: (16_729_200, 0),
        2015: (16_778_790, 0),
        2016: (16_785_700, 0),
        2017: (19_159_550, 0),
        2018: (19_096_040, 0),
        2019: (19_155_550, 0),
        2020: (23_725_480, 0),
        2021: (23_762_440, 0),
        2022: (23_810_850, 0),
        2023: (33_284_200, 0),
        2024: (33_195_740, 0),
    },
    ("Reynoldsburg", "ComInd"): {
        2014: (8_723_060, 24_367_000),
        2015: (8_243_170, 24_367_000),
        2016: (8_243_170, 24_395_040),
        2017: (10_733_250, 24_531_500),
        2018: (14_170_450, 24_531_500),
        2019: (15_314_850, 24_531_500),
        2020: (17_743_950, 25_936_270),
        2021: (13_750_060, 25_936_240),
        2022: (13_750_060, 25_936_240),
        2023: (11_455_830, 24_531_550),
        2024: (11_455_830, 24_531_550),
    },
    ("Columbus", "ResAgr"): {
        2014: (4_214_470, 0),
        2015: (4_573_250, 0),
        2016: (5_264_870, 0),
        2017: (6_131_530, 0),
        2018: (7_966_660, 0),
        2019: (9_122_090, 0),
        2020: (12_079_200, 0),
        2021: (12_103_750, 0),
        2022: (12_132_270, 0),
        2023: (17_026_390, 0),
        2024: (17_042_400, 0),
    }
}

# --- 2) Flatten
rows = []
for (muni, pclass), yearly_vals in taxable_data.items():
    for year, (v1, v2) in yearly_vals.items():
        rows.append({
            "TaxYear": year,
            "Municipality": muni,
            "PropertyClass": pclass,
            "TaxableValue": v1 + v2
        })
df = pd.DataFrame(rows)

# --- 3) Sum across municipalities -> township-wide by PropertyClass × year
township = (
    df.groupby(["TaxYear", "PropertyClass"], as_index=False)
      .agg(TaxableValue=("TaxableValue", "sum"))
      .sort_values(["PropertyClass", "TaxYear"])
)

# --- 4) YoY % change by PropertyClass (township totals)
township["TaxableValue_pct_change"] = (
    township.groupby("PropertyClass")["TaxableValue"].pct_change() * 100
)

# --- 5) Average % change per PropertyClass (exclude first NaN row)
avg_pct_change = (
    township.dropna(subset=["TaxableValue_pct_change"])
            .groupby("PropertyClass", as_index=False)
            .agg(avg_taxable_value_pct_change=("TaxableValue_pct_change", "mean"))
)

# --- 6) Print results
print("Township-wide taxable value (summed across municipalities) with YoY % change:")
print(township.to_string(index=False))

print("\nAverage YoY % change in taxable value by PropertyClass (township-wide):")
print(avg_pct_change.to_string(index=False))

Township-wide taxable value (summed across municipalities) with YoY % change:
 TaxYear PropertyClass  TaxableValue  TaxableValue_pct_change
    2014        ComInd      88075150                      NaN
    2015        ComInd      53801810               -38.913746
    2016        ComInd      50212460                -6.671430
    2017        ComInd      54885140                 9.305818
    2018        ComInd      65368890                19.101254
    2019        ComInd      62972550                -3.665872
    2020        ComInd      69420850                10.239858
    2021        ComInd      66975600                -3.522357
    2022        ComInd      67197330                 0.331061
    2023        ComInd      84023000                25.039194
    2024        ComInd      71725720               -14.635612
    2014        ResAgr     434286850                      NaN
    2015        ResAgr     416647710                -4.061633
    2016        ResAgr     422800830                 1

In [11]:
import pandas as pd
from pathlib import Path
import numpy as np

# --- 0) Paths (adjust ROOT if your notebook location differs)
ROOT = Path().resolve().parents[1]
MILLAGE_FP = ROOT / 'data' / 'Township_Millage_Table.csv'

# --- 1) Your taxable-value inputs -------------------------------------------
taxable_data = {
    ("Jefferson Unincorporated", "ResAgr"): {
        2014: (407_807_070, 2_731_540),
        2015: (389_535_970, 2_893_880),
        2016: (394_781_780, 3_114_460),
        2017: (448_618_560, 3_264_290),
        2018: (457_955_400, 3_368_530),
        2019: (477_750_550, 5_763_030),
        2020: (568_049_370, 3_308_040),
        2021: (585_038_010, 3_641_900),
        2022: (576_718_700, 3_603_720),
        2023: (768_096_650, 4_056_820),
        2024: (763_057_600, 3_177_510),
    },
    ("Jefferson Unincorporated", "ComInd"): {
        2014: (18_498_090, 36_028_230),
        2015: (8_989_220, 11_740_430),
        2016: (8_814_320, 8_106_700),
        2017: (9_816_490, 9_176_010),
        2018: (15_470_630, 9_214_410),
        2019: (11_671_440, 9_214_410),
        2020: (12_940_970, 10_286_530),
        2021: (13_639_040, 10_633_240),
        2022: (13_244_310, 10_148_800),
        2023: (26_236_970, 12_954_810),
        2024: (16_847_860, 12_925_410),
    },
    ("Gahanna", "ResAgr"): {
        2014: (2_705_600, 98_970),
        2015: (2_766_850, 98_970),
        2016: (2_755_050, 98_970),
        2017: (2_730_400, 99_530),
        2018: (2_823_610, 99_530),
        2019: (3_152_540, 99_530),
        2020: (4_121_040, 89_370),
        2021: (4_182_260, 8_990),
        2022: (4_165_320, 0),
        2023: (5_379_930, 0),
        2024: (5_347_560, 0),
    },
    ("Gahanna", "ComInd"): {
        2014: (457_120, 1_650),
        2015: (460_340, 1_650),
        2016: (651_580, 1_650),
        2017: (626_240, 1_650),
        2018: (1_980_250, 1_650),
        2019: (2_238_700, 1_650),
        2020: (2_511_240, 1_890),
        2021: (3_015_130, 1_890),
        2022: (3_514_270, 603_650),
        2023: (8_013_600, 830_240),
        2024: (5_418_120, 546_950),
    },
    ("Reynoldsburg", "ResAgr"): {
        2014: (16_729_200, 0),
        2015: (16_778_790, 0),
        2016: (16_785_700, 0),
        2017: (19_159_550, 0),
        2018: (19_096_040, 0),
        2019: (19_155_550, 0),
        2020: (23_725_480, 0),
        2021: (23_762_440, 0),
        2022: (23_810_850, 0),
        2023: (33_284_200, 0),
        2024: (33_195_740, 0),
    },
    ("Reynoldsburg", "ComInd"): {
        2014: (8_723_060, 24_367_000),
        2015: (8_243_170, 24_367_000),
        2016: (8_243_170, 24_395_040),
        2017: (10_733_250, 24_531_500),
        2018: (14_170_450, 24_531_500),
        2019: (15_314_850, 24_531_500),
        2020: (17_743_950, 25_936_270),
        2021: (13_750_060, 25_936_240),
        2022: (13_750_060, 25_936_240),
        2023: (11_455_830, 24_531_550),
        2024: (11_455_830, 24_531_550),
    },
    ("Columbus", "ResAgr"): {
        2014: (4_214_470, 0),
        2015: (4_573_250, 0),
        2016: (5_264_870, 0),
        2017: (6_131_530, 0),
        2018: (7_966_660, 0),
        2019: (9_122_090, 0),
        2020: (12_079_200, 0),
        2021: (12_103_750, 0),
        2022: (12_132_270, 0),
        2023: (17_026_390, 0),
        2024: (17_042_400, 0),
    }
}

# Flatten & sum across municipalities -> township totals by PropertyClass × year
rows = []
for (muni, pclass), yearly in taxable_data.items():
    for year, (v1, v2) in yearly.items():
        rows.append({"TaxYear": year, "Municipality": muni,
                     "PropertyClass": pclass, "TaxableValue": v1 + v2})
tax_df = pd.DataFrame(rows)
township = (tax_df.groupby(["TaxYear", "PropertyClass"], as_index=False)
                  .agg(TaxableValue=("TaxableValue", "sum"))
                  .sort_values(["PropertyClass", "TaxYear"]))
township["TaxableValue_pct_change"] = (
    township.groupby("PropertyClass")["TaxableValue"].pct_change() * 100
)

# --- 2) FireRate % changes collapsed by PropertyClass (same across munis) ---
millage = pd.read_csv(MILLAGE_FP)
need = ["TaxYear", "PropertyClass", "FireRate"]
if not set(need).issubset(millage.columns):
    raise ValueError(f"Millage file missing columns: {set(need) - set(millage.columns)}")

fire = (millage.groupby(["TaxYear", "PropertyClass"], as_index=False)
               .agg(FireRate=("FireRate", "mean"))
               .sort_values(["PropertyClass", "TaxYear"]))
fire["FireRate_pct_change"] = (
    fire.groupby("PropertyClass")["FireRate"].pct_change() * 100
)

# --- 3) Merge & compute elasticity -----------------------------------------
panel = pd.merge(
    fire[["TaxYear", "PropertyClass", "FireRate_pct_change"]],
    township[["TaxYear", "PropertyClass", "TaxableValue_pct_change"]],
    on=["TaxYear", "PropertyClass"],
    how="inner"
)

# Avoid divide-by-zero / undefined
panel["Elasticity"] = np.where(
    panel["TaxableValue_pct_change"].replace({0: np.nan}).notna(),
    panel["FireRate_pct_change"] / panel["TaxableValue_pct_change"],
    np.nan
)

# --- 4) Print per-year elasticities and averages ----------------------------
print("Per-year elasticity by PropertyClass (FireRate% / TaxableValue%):")
print(panel.sort_values(["PropertyClass", "TaxYear"])
           .to_string(index=False,
                      float_format=lambda x: f"{x:.6f}"))

avg_elasticity = (panel.dropna(subset=["Elasticity"])
                        .groupby("PropertyClass", as_index=False)
                        .agg(avg_elasticity=("Elasticity", "mean")))
print("\nAverage elasticity by PropertyClass (simple mean over available years):")
print(avg_elasticity.to_string(index=False,
                               float_format=lambda x: f"{x:.6f}"))

Per-year elasticity by PropertyClass (FireRate% / TaxableValue%):
 TaxYear PropertyClass  FireRate_pct_change  TaxableValue_pct_change  Elasticity
    2014        ComInd                  NaN                      NaN         NaN
    2015        ComInd            -9.265686               -38.913746    0.238108
    2016        ComInd             1.387812                -6.671430   -0.208023
    2017        ComInd            -3.189684                 9.305818   -0.342762
    2018        ComInd            -0.768317                19.101254   -0.040223
    2019        ComInd             1.124239                -3.665872   -0.306677
    2020        ComInd            -9.517588                10.239858   -0.929465
    2021        ComInd             4.956738                -3.522357   -1.407222
    2022        ComInd            18.758743                 0.331061   56.662520
    2023        ComInd           -13.203127                25.039194   -0.527298
    2024        ComInd             0.482493

In [15]:
import pandas as pd
from pathlib import Path
import numpy as np

# ---------- Paths ----------
ROOT = Path().resolve().parents[1]
DATA_DIR = ROOT / "data"
MILLAGE_FP = DATA_DIR / "Township_Millage_Table.csv"
TIF_FP     = DATA_DIR / "Jefferson_TIF_Details_All_Years.csv"

# ---------- Helper: classify PropertyClass ----------
def normalize_str(x):
    return str(x).strip().lower() if pd.notna(x) else ""

def classify_pc_from_row(row):
    # 1) Try TaxRateType text first
    trt = normalize_str(row.get("TaxRateType", ""))
    if trt:
        if any(k in trt for k in ["resagr", "res/agr", "residential", "agric"]):
            return "ResAgr"
        if any(k in trt for k in ["comind", "com/ind", "commercial", "industrial"]):
            return "ComInd"
    # 2) Fall back to TaxableLUC
    luc = pd.to_numeric(row.get("TaxableLUC", np.nan), errors="coerce")
    if pd.notna(luc):
        if 100 <= int(luc) <= 299:
            return "ResAgr"
        if 300 <= int(luc) <= 499:
            return "ComInd"
    # 3) Fall back to TIFLUC
    luc2 = pd.to_numeric(row.get("TIFLUC", np.nan), errors="coerce")
    if pd.notna(luc2):
        if 100 <= int(luc2) <= 299:
            return "ResAgr"
        if 300 <= int(luc2) <= 499:
            return "ComInd"
    # Unknown
    return np.nan

# ---------- Load data ----------
millage = pd.read_csv(MILLAGE_FP)
tif = pd.read_csv(TIF_FP)

# ---------- Municipality from TaxDistrict ----------
tif["TaxDistrict_str"] = tif["TaxDistrict"].astype(str).str.zfill(3)
td_to_muni = {
    "170": "Jefferson Unincorporated",
    "171": "Jefferson Unincorporated",
    "027": "Gahanna",
    "067": "Reynoldsburg",
    "175": "Columbus",
}
tif["Municipality"] = tif["TaxDistrict_str"].map(td_to_muni).fillna("Other")

# ---------- Classify PropertyClass robustly ----------
tif["PropertyClass"] = tif.apply(classify_pc_from_row, axis=1)

# ---------- Quick diagnostics ----------
print("\n=== Diagnostics ===")
print("TIF rows total:", len(tif))
print("Municipality value counts (top 10):")
print(tif["Municipality"].value_counts().head(10).to_string())

print("\nPropertyClass value counts BEFORE dropping unknowns:")
print(tif["PropertyClass"].value_counts(dropna=False).to_string())

unknown_pc = tif[tif["PropertyClass"].isna()]
if not unknown_pc.empty:
    print("\nSample of rows with unknown PropertyClass (up to 10):")
    cols_preview = ["TaxYear","TaxDistrict_str","CityVillage","TaxRateType","TaxableLUC","TIFLUC","AssessedImpr"]
    print(unknown_pc[cols_preview].head(10).to_string(index=False))

# ---------- Keep only mapped municipalities & known classes ----------
tif_clean = tif[(tif["Municipality"] != "Other") & tif["PropertyClass"].notna()].copy()

# ---------- Township taxable totals you provided (unchanged) ----------
taxable_data = {
    ("Jefferson Unincorporated", "ResAgr"): {
        2014: (407_807_070, 2_731_540), 2015: (389_535_970, 2_893_880),
        2016: (394_781_780, 3_114_460), 2017: (448_618_560, 3_264_290),
        2018: (457_955_400, 3_368_530), 2019: (477_750_550, 5_763_030),
        2020: (568_049_370, 3_308_040), 2021: (585_038_010, 3_641_900),
        2022: (576_718_700, 3_603_720), 2023: (768_096_650, 4_056_820),
        2024: (763_057_600, 3_177_510),
    },
    ("Jefferson Unincorporated", "ComInd"): {
        2014: (18_498_090, 36_028_230), 2015: (8_989_220, 11_740_430),
        2016: (8_814_320, 8_106_700),  2017: (9_816_490, 9_176_010),
        2018: (15_470_630, 9_214_410), 2019: (11_671_440, 9_214_410),
        2020: (12_940_970, 10_286_530),2021: (13_639_040, 10_633_240),
        2022: (13_244_310, 10_148_800),2023: (26_236_970, 12_954_810),
        2024: (16_847_860, 12_925_410),
    },
    ("Gahanna", "ResAgr"): {
        2014: (2_705_600, 98_970),   2015: (2_766_850, 98_970),
        2016: (2_755_050, 98_970),   2017: (2_730_400, 99_530),
        2018: (2_823_610, 99_530),   2019: (3_152_540, 99_530),
        2020: (4_121_040, 89_370),   2021: (4_182_260, 8_990),
        2022: (4_165_320, 0),        2023: (5_379_930, 0),
        2024: (5_347_560, 0),
    },
    ("Gahanna", "ComInd"): {
        2014: (457_120, 1_650),    2015: (460_340, 1_650),
        2016: (651_580, 1_650),    2017: (626_240, 1_650),
        2018: (1_980_250, 1_650),  2019: (2_238_700, 1_650),
        2020: (2_511_240, 1_890),  2021: (3_015_130, 1_890),
        2022: (3_514_270, 603_650),2023: (8_013_600, 830_240),
        2024: (5_418_120, 546_950),
    },
    ("Reynoldsburg", "ResAgr"): {
        2014: (16_729_200, 0), 2015: (16_778_790, 0), 2016: (16_785_700, 0),
        2017: (19_159_550, 0), 2018: (19_096_040, 0), 2019: (19_155_550, 0),
        2020: (23_725_480, 0), 2021: (23_762_440, 0), 2022: (23_810_850, 0),
        2023: (33_284_200, 0), 2024: (33_195_740, 0),
    },
    ("Reynoldsburg", "ComInd"): {
        2014: (8_723_060, 24_367_000), 2015: (8_243_170, 24_367_000),
        2016: (8_243_170, 24_395_040), 2017: (10_733_250, 24_531_500),
        2018: (14_170_450, 24_531_500), 2019: (15_314_850, 24_531_500),
        2020: (17_743_950, 25_936_270), 2021: (13_750_060, 25_936_240),
        2022: (13_750_060, 25_936_240), 2023: (11_455_830, 24_531_550),
        2024: (11_455_830, 24_531_550),
    },
    ("Columbus", "ResAgr"): {
        2014: (4_214_470, 0), 2015: (4_573_250, 0), 2016: (5_264_870, 0),
        2017: (6_131_530, 0), 2018: (7_966_660, 0), 2019: (9_122_090, 0),
        2020: (12_079_200, 0),2021: (12_103_750, 0),2022: (12_132_270, 0),
        2023: (17_026_390, 0),2024: (17_042_400, 0),
    }
}

rows = []
for (muni, pcls), yearly in taxable_data.items():
    for year, (v1, v2) in yearly.items():
        rows.append({"TaxYear": year, "Municipality": muni,
                     "PropertyClass": pcls, "TaxableValue": v1 + v2})
tax_df = pd.DataFrame(rows)
township_totals = (tax_df.groupby(["TaxYear","PropertyClass"], as_index=False)
                          .agg(Total_Township_TaxableValue=("TaxableValue","sum")))

# ---------- Fire rates collapsed by PropertyClass ----------
need_cols = {"TaxYear","PropertyClass","FireRate"}
if not need_cols.issubset(millage.columns):
    raise ValueError(f"Millage file missing columns: {need_cols - set(millage.columns)}")
fire_rates = (millage.groupby(["TaxYear","PropertyClass"], as_index=False)
                     .agg(FireRate=("FireRate","mean")))

# ---------- Elasticities ----------
elasticity_map = {"ComInd": 5.310599, "ResAgr": -2.349367}

# ---------- Aggregate TIF base ----------
tif_muni = (tif_clean.groupby(["TaxYear","Municipality","PropertyClass"], as_index=False)
                      .agg(TIF_AssessedImpr=("AssessedImpr","sum")))
tif_township = (tif_muni.groupby(["TaxYear","PropertyClass"], as_index=False)
                        .agg(TIF_Township_AssessedImpr=("TIF_AssessedImpr","sum")))

# ---------- Merge & compute ----------
panel = (tif_muni
         .merge(township_totals, on=["TaxYear","PropertyClass"], how="left")
         .merge(tif_township, on=["TaxYear","PropertyClass"], how="left")
         .merge(fire_rates, on=["TaxYear","PropertyClass"], how="left"))

panel["TIF_Share_of_Base"] = panel["TIF_Township_AssessedImpr"] / panel["Total_Township_TaxableValue"]
panel["Elasticity"] = panel["PropertyClass"].map(elasticity_map).astype(float)
panel["FireRate_New"] = panel["FireRate"] * (1 + panel["Elasticity"] * panel["TIF_Share_of_Base"])

def levy_amount(mills, assessed):  # mills per $1000 assessed
    return mills * (assessed / 1000.0)

panel["Direct_Loss_Baseline"] = levy_amount(panel["FireRate"],     panel["TIF_AssessedImpr"])
panel["Direct_Loss_AdjRate"]  = levy_amount(panel["FireRate_New"], panel["TIF_AssessedImpr"])

# Township rate effect and allocation
town_rate = (panel.groupby(["TaxYear","PropertyClass"], as_index=False).first()[["TaxYear","PropertyClass"]]
            ).merge(fire_rates, on=["TaxYear","PropertyClass"], how="left"
            ).merge(tif_township, on=["TaxYear","PropertyClass"], how="left"
            ).merge(township_totals, on=["TaxYear","PropertyClass"], how="left")
town_rate["Elasticity"] = town_rate["PropertyClass"].map(elasticity_map)
town_rate["Share"] = town_rate["TIF_Township_AssessedImpr"] / town_rate["Total_Township_TaxableValue"]
town_rate["FireRate_New"] = town_rate["FireRate"] * (1 + town_rate["Elasticity"] * town_rate["Share"])
town_rate["DeltaRate"] = town_rate["FireRate_New"] - town_rate["FireRate"]
town_rate["Township_RateEffect_RevenueChange"] = levy_amount(
    town_rate["DeltaRate"], town_rate["Total_Township_TaxableValue"]
)

alloc_key = panel.groupby(["TaxYear","PropertyClass"], as_index=False)\
                 .agg(TIF_Township_AssessedImpr=("TIF_AssessedImpr","sum"))
panel = panel.merge(alloc_key, on=["TaxYear","PropertyClass"], suffixes=("","_TownshipCheck"))
panel["TIF_ShareWithinPC"] = np.where(
    panel["TIF_Township_AssessedImpr"] > 0,
    panel["TIF_AssessedImpr"] / panel["TIF_Township_AssessedImpr"],
    0.0
)
panel = panel.merge(
    town_rate[["TaxYear","PropertyClass","Township_RateEffect_RevenueChange"]],
    on=["TaxYear","PropertyClass"], how="left"
)
panel["Allocated_RateEffect_RevenueChange"] = panel["TIF_ShareWithinPC"] * panel["Township_RateEffect_RevenueChange"]

# ---------- Output ----------
print("\n=== Per-PropertyClass details (head) ===")
print(panel[["TaxYear","Municipality","PropertyClass","FireRate","FireRate_New",
             "TIF_AssessedImpr","TIF_Township_AssessedImpr","Total_Township_TaxableValue",
             "TIF_Share_of_Base","Elasticity",
             "Direct_Loss_Baseline","Direct_Loss_AdjRate",
             "Allocated_RateEffect_RevenueChange"]]
      .sort_values(["TaxYear","Municipality","PropertyClass"])
      .head(20)
      .to_string(index=False, float_format=lambda x: f"{x:,.2f}"))

muni_year = (panel.groupby(["TaxYear","Municipality"], as_index=False)
                  .agg(
                      Direct_Loss_Baseline=("Direct_Loss_Baseline","sum"),
                      Direct_Loss_AdjRate=("Direct_Loss_AdjRate","sum"),
                      Allocated_RateEffect_RevenueChange=("Allocated_RateEffect_RevenueChange","sum")
                  ))
muni_year["Net_BaselinePlusRateEffect"] = (
    muni_year["Direct_Loss_Baseline"] + muni_year["Allocated_RateEffect_RevenueChange"]
)

print("\n=== Municipality × Year totals ===")
print(muni_year.sort_values(["TaxYear","Municipality"])
      .to_string(index=False, float_format=lambda x: f"{x:,.2f}"))


=== Diagnostics ===
TIF rows total: 6370
Municipality value counts (top 10):
Municipality
Jefferson Unincorporated    3195
Columbus                    2354
Gahanna                      728
Reynoldsburg                  93

PropertyClass value counts BEFORE dropping unknowns:
PropertyClass
NaN       6268
ComInd     102

Sample of rows with unknown PropertyClass (up to 10):
 TaxYear TaxDistrict_str   CityVillage TaxRateType                           TaxableLUC  TIFLUC  AssessedImpr
    2014             175 COLUMBUS CITY      Res/Ag 510 - ONE-FAMILY DWLG ON PLATTED LOT   720.0       32550.0
    2014             175 COLUMBUS CITY      Res/Ag 510 - ONE-FAMILY DWLG ON PLATTED LOT   720.0       33530.0
    2014             175 COLUMBUS CITY      Res/Ag 510 - ONE-FAMILY DWLG ON PLATTED LOT   720.0       31010.0
    2014             175 COLUMBUS CITY      Res/Ag 510 - ONE-FAMILY DWLG ON PLATTED LOT   720.0       31470.0
    2014             175 COLUMBUS CITY      Res/Ag 510 - ONE-FAMILY DWLG O